# Tokenizers: BPE, WordPiece, SentencePiece Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Character-Level Tokenizer

Start at the foundation. A character-level tokenizer maps each character to its Unicode code point. No training needed. No unknown tokens. Just a direct mapping.

In [ ]:
```python

class CharTokenizer:

    def encode(self, text):

        return [ord(c) for c in text]

    def decode(self, tokens):

        return "".join(chr(t) for t in tokens)

In [ ]:
```

"hello" becomes [104, 101, 108, 108, 111]. Every character is its own token. This is the baseline we improve on.

### Step 2: BPE Tokenizer from Scratch

The real implementation. We train on raw bytes (like GPT-2), count pairs, merge the most frequent, and record every merge in order. The merge table is the tokenizer.

In [ ]:
```python

from collections import Counter

class BPETokenizer:

    def __init__(self):

        self.merges = {}

        self.vocab = {}

    def _get_pairs(self, tokens):

        pairs = Counter()

        for i in range(len(tokens) - 1):

            pairs[(tokens[i], tokens[i + 1])] += 1

        return pairs

    def _merge_pair(self, tokens, pair, new_token):

        merged = []

        i = 0

        while i < len(tokens):

            if i < len(tokens) - 1 and tokens[i] == pair[0] and tokens[i + 1] == pair[1]:

                merged.append(new_token)

                i += 2

            else:

                merged.append(tokens[i])

                i += 1

        return merged

    def train(self, text, num_merges):

        tokens = list(text.encode("utf-8"))

        self.vocab = {i: bytes([i]) for i in range(256)}

        for i in range(num_merges):

            pairs = self._get_pairs(tokens)

            if not pairs:

                break

            best_pair = max(pairs, key=pairs.get)

            new_token = 256 + i

            tokens = self._merge_pair(tokens, best_pair, new_token)

            self.merges[best_pair] = new_token

            self.vocab[new_token] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]

        return self

    def encode(self, text):

        tokens = list(text.encode("utf-8"))

        for pair, new_token in self.merges.items():

            tokens = self._merge_pair(tokens, pair, new_token)

        return tokens

    def decode(self, tokens):

        byte_sequence = b"".join(self.vocab[t] for t in tokens)

        return byte_sequence.decode("utf-8", errors="replace")

In [ ]:
```

The training loop is the core of BPE: count pairs, merge the winner, repeat. Each merge reduces the total token count. After `num_merges` rounds, the vocabulary grows from 256 (base bytes) to 256 + num_merges.

Encoding applies merges in the exact order they were learned. This matters. If merge 1 created "th" and merge 5 created "the", encoding must apply merge 1 first so that "the" can form from "th" + "e" in merge 5.

Decoding is the inverse: look up each token ID in the vocabulary, concatenate the bytes, decode to UTF-8.

### Step 3: Encode and Decode Roundtrip

In [ ]:
```python

corpus = (

    "The cat sat on the mat. The cat ate the rat. "

    "The dog sat on the log. The dog ate the frog. "

    "Natural language processing is the study of how computers "

    "understand and generate human language. "

    "Tokenization is the first step in any NLP pipeline."

)

tokenizer = BPETokenizer()

tokenizer.train(corpus, num_merges=40)

test_sentences = [

    "The cat sat on the mat.",

    "Natural language processing",

    "tokenization pipeline",

    "unhappiness",

]

for sentence in test_sentences:

    encoded = tokenizer.encode(sentence)

    decoded = tokenizer.decode(encoded)

    raw_bytes = len(sentence.encode("utf-8"))

    ratio = len(encoded) / raw_bytes

    print(f"'{sentence}'")

    print(f"  Tokens: {len(encoded)} (from {raw_bytes} bytes) -- ratio: {ratio:.2f}")

    print(f"  Roundtrip: {'PASS' if decoded == sentence else 'FAIL'}")

In [ ]:
```

The compression ratio tells you how effective the tokenizer is. A ratio of 0.50 means the tokenizer compressed the text to half as many tokens as raw bytes. Lower is better. On the training corpus, the ratio will be good. On out-of-distribution text like "unhappiness" (which does not appear in the corpus), the ratio will be worse -- the tokenizer falls back to character-level encoding for unseen patterns.

### Step 4: Compare with tiktoken

In [ ]:
```python

import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

texts = [

    "The cat sat on the mat.",

    "unhappiness",

    "Hello, world!",

    "def fibonacci(n): return n if n < 2 else fibonacci(n-1) + fibonacci(n-2)",

    "Geschwindigkeitsbegrenzung",

]

for text in texts:

    our_tokens = tokenizer.encode(text)

    tiktoken_tokens = enc.encode(text)

    tiktoken_pieces = [enc.decode([t]) for t in tiktoken_tokens]

    print(f"'{text}'")

    print(f"  Our BPE:   {len(our_tokens)} tokens")

    print(f"  tiktoken:  {len(tiktoken_tokens)} tokens -> {tiktoken_pieces}")

In [ ]:
```

tiktoken uses the exact same algorithm but trained on hundreds of gigabytes of text with 100,000 merges. The algorithm is identical. The difference is the training data and the number of merges. Your tokenizer trained on a paragraph with 40 merges cannot compete with tiktoken's 100K merges on a massive corpus. But the mechanism is the same.

### Step 5: Vocabulary Analysis

In [ ]:
```python

def analyze_vocabulary(tokenizer, test_texts):

    total_tokens = 0

    total_chars = 0

    token_usage = Counter()

    for text in test_texts:

        encoded = tokenizer.encode(text)

        total_tokens += len(encoded)

        total_chars += len(text)

        for t in encoded:

            token_usage[t] += 1

    print(f"Vocabulary size: {len(tokenizer.vocab)}")

    print(f"Total tokens across all texts: {total_tokens}")

    print(f"Total characters: {total_chars}")

    print(f"Avg tokens per character: {total_tokens / total_chars:.2f}")

    print(f"\nMost used tokens:")

    for token_id, count in token_usage.most_common(10):

        token_bytes = tokenizer.vocab[token_id]

        display = token_bytes.decode("utf-8", errors="replace")

        print(f"  Token {token_id:4d}: '{display}' (used {count} times)")

    unused = [t for t in tokenizer.vocab if t not in token_usage]

    print(f"\nUnused tokens: {len(unused)} out of {len(tokenizer.vocab)}")

In [ ]:
```

This reveals the Zipf distribution in your vocabulary. A few tokens dominate (spaces, "the", "e"). Most tokens are rarely used. Production tokenizers optimize for this distribution -- common patterns get short token IDs, rare patterns get longer representations.

## Exercises

In [ ]:
1. Modify the BPE tokenizer to print the vocabulary at each merge step. Watch how "t" + "h" becomes "th", then "th" + "e" becomes "the". Track how common English words get assembled piece by piece.

2. Add special tokens (`<pad>`, `<eos>`, `<unk>`) to the BPE tokenizer. Assign them IDs 0, 1, 2 and shift all other tokens accordingly. Implement a pre-tokenization step that splits on whitespace before running BPE.

3. Implement the WordPiece merge criterion (likelihood ratio instead of frequency). Train both BPE and WordPiece on the same corpus with the same number of merges. Compare the resulting vocabularies -- which one produces more linguistically meaningful subwords?

4. Build a multilingual tokenizer efficiency benchmark. Take 10 sentences in English, Spanish, Chinese, Korean, and Arabic. Tokenize each with tiktoken (cl100k_base) and measure the average tokens per character. Quantify the "multilingual tax" for each language.

5. Train your BPE tokenizer on a larger corpus (download a Wikipedia article). Tune the number of merges to achieve a compression ratio within 10% of tiktoken on that same text. This forces you to understand the relationship between corpus size, merge count, and compression quality.